<a href="https://colab.research.google.com/github/carloshsieh22/musicbehindthemedal/blob/main/DATASCI_112_Final_Project_Carlos_Allison_Model_Building.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XG Boost Regressor Model to Test the Impact of Song Choice

We trained 2 different models, one with song genre as a feature, another without. The models predicted PCS (program component score) and measured error using negative mean squared error.



In [ ]:
import pandas as pd
combined_df = pd.read_csv('combined_df.csv')
combined_df = combined_df.drop(['Unnamed: 0'], axis=1)
free_skate_songs = pd.read_csv('free_skate_songs.csv')
short_program_songs = pd.read_csv('short_program_songs.csv')

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

Linear Regression

In [ ]:
transformer_with = make_column_transformer(
    (OneHotEncoder(sparse_output=False, handle_unknown="ignore"), ["Nation", "Competition", "Gender", "sp_category"]),
    (StandardScaler(), ["SP Score", "SP Rank", "SP TES", "SP PR", "SP CO"])
)

transformer_without = make_column_transformer(
    (OneHotEncoder(sparse_output=False, handle_unknown="ignore"), ["Nation", "Competition", "Gender"]),
    (StandardScaler(), ["SP Score", "SP Rank", "SP TES", "SP PR", "SP CO"])
)
pipeline_with = make_pipeline(transformer_with,
                              LinearRegression())
pipeline_without = make_pipeline(transformer_without,
                                 LinearRegression())
X_train_with = combined_df[['Competition', 'Nation', 'SP Score', 'SP Rank', 'SP TES',
       'SP PR', 'SP CO', 'Gender', 'sp_category']]
X_train_without = combined_df[['Competition', 'Nation', 'SP Score', 'SP Rank', 'SP TES',
       'SP PR', 'SP CO', 'Gender']]
y_train = combined_df['SP PCS']
rmse_with    = cross_val_score(pipeline_with, X=X_train_with, y=y_train, cv=5, scoring="neg_root_mean_squared_error")
rmse_without = cross_val_score(pipeline_without, X=X_train_without, y=y_train, cv=5, scoring="neg_root_mean_squared_error")

In [ ]:
print(np.sqrt(-rmse_with.mean()))
print(np.sqrt(-rmse_without.mean()))

0.7147144836403581
0.7133761465371848


Initial XGBoost cross validation scores XGBoost to capture non linear relationship.

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

transformer_with = make_column_transformer(
    (OneHotEncoder(sparse_output=False, handle_unknown="ignore"), ["Nation", "Competition", "Gender", "sp_category"]),
    (StandardScaler(), ["SP Score", "SP Rank", "SP TES", "SP PR", "SP CO"])
)

transformer_without = make_column_transformer(
    (OneHotEncoder(sparse_output=False, handle_unknown="ignore"), ["Nation", "Competition", "Gender"]),
    (StandardScaler(), ["SP Score", "SP Rank", "SP TES", "SP PR", "SP CO"])
)

# ── Pipelines ─────────────────────────────────────────────────────────────────
pipeline_with    = make_pipeline(transformer_with,    XGBRegressor())
pipeline_without = make_pipeline(transformer_without, XGBRegressor())

# ── Grid search params ────────────────────────────────────────────────────────
param_grid = {
    "xgbregressor__n_estimators":  [100, 200, 300],
    "xgbregressor__max_depth":     [3, 4, 5],
    "xgbregressor__learning_rate": [0.01, 0.05, 0.1],
    "xgbregressor__subsample":     [0.8, 1.0],
}

# ── Grid search ───────────────────────────────────────────────────────────────
rmse_with    = cross_val_score(pipeline_with, X=X_train_with, y=y_train, cv=5, scoring="neg_root_mean_squared_error")
rmse_without = cross_val_score(pipeline_without, X=X_train_without, y=y_train, cv=5, scoring="neg_root_mean_squared_error")
print(np.sqrt(-rmse_with.mean()))
print(np.sqrt(-rmse_without.mean()))

0.9467133122209783
0.9460473325111534


Hyperparameter Searching

In [ ]:
gs_with = GridSearchCV(
    pipeline_with,
    param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

gs_without = GridSearchCV(
    pipeline_without,
    param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

gs_with.fit(X_train_with, y_train)
gs_without.fit(X_train_without, y_train)
print(f"\n  Best params with song:\n    {gs_with.best_params_}")
print(f"\n  Best params without song:\n    {gs_without.best_params_}")

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Fitting 5 folds for each of 54 candidates, totalling 270 fits
  Best RMSE with song:    0.6958
  Best RMSE without song: 0.6921
  Δ RMSE (with - without): +0.0037

  Best params with song:
    {'xgbregressor__learning_rate': 0.1, 'xgbregressor__max_depth': 3, 'xgbregressor__n_estimators': 300, 'xgbregressor__subsample': 0.8}

  Best params without song:
    {'xgbregressor__learning_rate': 0.1, 'xgbregressor__max_depth': 3, 'xgbregressor__n_estimators': 300, 'xgbregressor__subsample': 0.8}


In [ ]:
y_pred_with    = cross_val_predict(pipeline_with, X_train_with, y_train, cv=5)
y_pred_without = cross_val_predict(pipeline_without, X_train_without, y_train, cv=5)

NameError: name 'cross_val_predict' is not defined

Creating an actual vs prediction scatterplot graph with both models side by side.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    shared_xaxes=False,
    subplot_titles=("With song category", "Without song category"),
    horizontal_spacing=0.05,
)
min_val, max_val = float(y_train.min()), float(y_train.max())
for col, (y_pred, color, name) in enumerate([
    (y_pred_with,    "#E05C3A", "With song category"),
    (y_pred_without, "#5B8DB8", "Without song category"),
], start=1):
    fig.add_trace(go.Scatter(
        x=y_train.values,
        y=y_pred,
        mode="markers",
        name=name,
        marker=dict(color=color, size=5, opacity=0.6),
        hovertemplate="Actual: %{x:.2f}<br>Predicted: %{y:.2f}<extra></extra>",
        showlegend=False,
    ), row=1, col=col)

    # Perfect prediction line per panel
    fig.add_trace(go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode="lines",
        line=dict(color="black", width=2, dash="dash"),
        name="Perfect prediction",
        showlegend=(col == 1),
    ), row=1, col=col)

fig.update_layout(
    title=dict(
        text="Actual vs Predicted Short Program PCS",
        font=dict(size=20),
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    hovermode="closest",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.update_xaxes(title_text="Actual Short Program PCS", showgrid=True, gridcolor="#eeeeee")
fig.update_yaxes(title_text="Predicted Short Program PCS", showgrid=True, gridcolor="#eeeeee", col=1)
fig.update_yaxes(title_text="Predicted Short Program PCS", showgrid=True, gridcolor="#eeeeee", col=2)

fig.show()